In [1]:

from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.cluster import DBSCAN

from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy

from sklearn.decomposition import TruncatedSVD
import numpy as np
import seaborn as sns

In [3]:
processed_dir = Path("../data/processed")
results_dir = Path("../results/clustering")
model_dir = Path("../models")

matrice_tfidf_bigrammes = load_npz(processed_dir / "tfidf_matrix_bigrammes.npz")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
data = pd.read_csv(processed_dir / "sample_clean.csv")

assert matrice_tfidf_bigrammes.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(data["id"])

In [4]:
df = data.copy()

In [5]:
df

,id,products,advantages,description_texte,details,fabric,weight,pays
0,1,Active classic boxers,There's a reason why our boxers are a cult fav...,active classic boxers - there s a reason why o...,"silky capilene 1 fabric is ultralight, breatha...",3.7-oz 100% all-recycled polyester with gladio...,99 g 3.5 oz,Mexico
1,2,Active sport boxer briefs,Skinning up Glory requires enough movement wit...,active sport boxer briefs - skinning up glory ...,"stretch mesh provides support, open-weave mesh...","4.6-oz 93% polyester 71% recycled /7% spandex,...",60 g 2.1 oz,Israel
2,3,Active sport briefs,These superbreathable no-fly briefs are the mi...,active sport briefs - these superbreathable no...,"stretch mesh provides support, open-weave mesh...","4.6-oz 93% polyester 71% recycled /7% spandex,...",49 g 1.7 oz,Israel
3,4,Alpine guide pants,"Skin in, climb ice, switch to rock, traverse a...","alpine guide pants - skin in, climb ice, switc...",durable stretch-woven polyester 47% recycled w...,7.6-oz 90-denier 92% polyester 47% recycled /8...,597 g 20.7 oz,Vietnam
4,5,Alpine wind jkt,"On high ridges, steep ice and anything alpine,...","alpine wind jkt - on high ridges, steep ice an...","lightweight, breathable polyester ripstop fabr...",shell 1.3-oz 20-denier 100% polyester ripstop....,331 g 11.5 oz,China
...,...,...,...,...,...,...,...,...
495,496,Cap 2 bottoms,Cut loose from the maddening crowds and search...,cap 2 bottoms - cut loose from the maddening c...,capilene 2 fabric has an open knit that invite...,4-oz 100% polyester 54% recycled with gladiodo...,155 g 5.5 oz,Mexico
496,497,Cap 2 crew,This crew takes the edge off fickle weather. I...,cap 2 crew - this crew takes the edge off fick...,capilene 2 fabric has an open knit that invite...,4-oz 100% polyester 54% recycled with gladiodo...,170 g 6 oz,Mexico
497,498,All-time shell,No need to use that morning Times as an umbrel...,all-time shell - no need to use that morning t...,2.5-layer 100% all-recycled polyester shell wi...,"shell 2.5-layer, 4.1-oz 50-denier 100% all-rec...",493 g 17.4 oz,China
498,499,All-wear cargo shorts,All-Wear Cargo Shorts bask in the glory of swe...,all-wear cargo shorts - all-wear cargo shorts ...,lightweight organic cotton canvas zip fly with...,5.2-oz 100% organic cotton canvas. recyclable ...,349 g 12.1 oz,Thailand


In [6]:
X = matrice_tfidf_bigrammes

# apply truncated SVD

In [7]:
svd_models = {}
lsa_results = {}

for k in (10, 15, 20):
    svd = TruncatedSVD(n_components=k, random_state=42)
    lsa_results[k] = svd.fit_transform(X)
    svd_models[k] = svd

In [8]:
for k, projections in lsa_results.items():
    labels = projections.argmax(axis=1)
    counts = np.bincount(labels, minlength=k)

    print(f"\n{k} composantes")
    print(f"Variance expliquée : {svd_models[k].explained_variance_ratio_.sum():.3f}")
    print(f"Effectifs par composante : {counts}")
    print(f"Proportion affectée à la première : {(labels == 0).mean():.1%}")


10 composantes
Variance expliquée : 0.184
Effectifs par composante : [220  78  54  28   0   8  21  30  50  11]
Proportion affectée à la première : 44.0%

15 composantes
Variance expliquée : 0.235
Effectifs par composante : [200  69  47  28   0   8  20  31  34  12   0  25   7   8  11]
Proportion affectée à la première : 40.0%

20 composantes
Variance expliquée : 0.276
Effectifs par composante : [189  58  47  22   0   8  18  30  36  12   0  15   6  15   7   7   6  20
   2   2]
Proportion affectée à la première : 37.8%


In [9]:
lsa_results[10].shape, lsa_results[15].shape, lsa_results[20].shape

((500, 10), (500, 15), (500, 20))

In [10]:
topic_encoded_df_10k = pd.DataFrame(lsa_results[10], columns = [f"topic_{i+1}"for i in range(10)], index = df.index)
topic_encoded_df_10k["keywords"] = df['products']

In [11]:
topic_encoded_df_15k = pd.DataFrame(lsa_results[15], columns = [f"topic_{i+1}"for i in range(15)], index = df.index)
topic_encoded_df_15k["keywords"] = df['products']


In [12]:
topic_encoded_df_20k = pd.DataFrame(lsa_results[20], columns = [f"topic_{i+1}"for i in range(20)], index = df.index)
topic_encoded_df_20k["keywords"] = df['products']


In [13]:
# Assign each document to the topic it is the most linked to
topic_encoded_df_10k["composante"] = [np.argmax(doc) for doc in lsa_results[10]]
topic_encoded_df_10k["composante"].value_counts()

composante
0    220
1     78
2     54
8     50
7     30
3     28
6     21
9     11
5      8
Name: count, dtype: int64

In [14]:
topic_encoded_df_15k["composante"] = [np.argmax(doc) for doc in lsa_results[15]]
topic_encoded_df_15k["composante"].value_counts()

composante
0     200
1      69
2      47
8      34
7      31
3      28
11     25
6      20
9      12
14     11
5       8
13      8
12      7
Name: count, dtype: int64

In [15]:
topic_encoded_df_20k["composante"] = [np.argmax(doc) for doc in lsa_results[20]]
topic_encoded_df_20k["composante"].value_counts()

composante
0     189
1      58
2      47
8      36
7      30
3      22
17     20
6      18
11     15
13     15
9      12
5       8
14      7
15      7
16      6
12      6
18      2
19      2
Name: count, dtype: int64

La SVD en 20 topics ne permet pas des clusters de minimum 6 produits. On va donc se concentrer sur la SVD en 10 ou 15 topics.

In [16]:
display(topic_encoded_df_10k.head(1))
display(topic_encoded_df_15k.head(1))

,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,keywords,composante
0,0.230099,0.152658,0.179087,0.066671,-0.158123,0.014354,0.032439,0.013133,-0.064786,-0.089469,Active classic boxers,0


,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,topic_11,topic_12,topic_13,topic_14,topic_15,keywords,composante
0,0.230099,0.152663,0.179142,0.066728,-0.15697,0.014875,0.038065,0.014098,-0.070299,-0.086725,0.039523,0.005859,-0.045202,-0.023414,-0.036777,Active classic boxers,0


# évaluer le truncateSVD

In [24]:
top = df[["id", "products"]].copy()
top["score_topic_5"] = lsa_results[15][:, 5]  # 4 = 5e colonne
display(top.nlargest(5, "score_topic_5"))

,id,products,score_topic_5
237,238,Wyoming climbing poster,0.962042
370,371,Sticks 'n stones morocco poster,0.950952
47,48,Going big in b.c. poster,0.950389
489,490,Flyfishing the athabasca poster,0.942535
68,69,Lead an examined life poster,0.935263
